# Calibrate comprehensive catalogue

In [1]:
%reload_ext autoreload
%autoreload 2

In [147]:
# General library imports
import sys
import os
import numpy as np
from astropy.io import fits
from astropy.nddata import bitmask

In [178]:
from sp_validation import run_calibrate_cat as calibrate
from sp_validation import util
from sp_validation.basic import metacal
from sp_validation import calibration

In [149]:
obj = calibrate.CalibrateCat()

In [150]:
obj._params["input_path"] = "unions_shapepipe_comprehensive_2024_v1.4.2.fits"

In [151]:
dat = obj.read_cat()

In [152]:
print(f"Found {len(dat)} (~{util.millify(len(dat))}) objects in catalogue")

Found 5896744 (~6 Million) objects in catalogue


## Masking

## Pre-processing ShapePipe flags

In [153]:
cut_pre = {}

In [154]:
sum(dat["FLAGS"] == 0)

4835857

In [155]:
# SExtractor flags (see galaxy.py:classification_galaxy_base)

name = "FLAGS"
good_mask_value = 0

# MKDBEUG TODO: implement values other than 0 as "good"

cut_pre[name] = (dat[name] == good_mask_value)

In [156]:
# Duplicate objects

name = "overlap"
good_mask_value = False
cut_pre[name] = bitmask.bitfield_to_boolean_mask(
    dat[name],
    good_mask_value=good_mask_value,
    dtype=bool,
)

In [157]:
# ShapePipe mask
name = "IMAFLAGS_ISO"
good_mask_value = 0
cut_pre[name] = (dat[name] == good_mask_value)

#bitmask.bitfield_to_boolean_mask(
#    dat[name],
#    good_mask_value=good_mask_value,
#    dtype=bool,
#)

In [158]:
# Number of epochs
name = "N_EPOCH"
val_min = 2
cut_pre[name] = (dat[name] >= val_min)

# MKDEBUG check NGMIX_N_EPOCH

In [159]:
# Magnitude range
name = "mag"
min_max = [15, 30]
cut_pre[name] = (
    (dat[name] >= min_max[0])
    & (dat[name] <= min_max[1])
)

In [160]:
# ngmix flags
names = ["NGMIX_MCAL_FLAGS", "NGMIX_MOM_FAIL"]
good_mask_values = [0, 0]
for name, good_mask_value in zip(names, good_mask_values):
    cut_pre[name] = (dat[name] == good_mask_value) 
    
name = "NGMIX_ELL_PSFo_NOSHEAR_0"
bad_mask_value = -10
cut_pre[name] = (
    dat[name] != bad_mask_value
)
# MKDEBUG TODO: check should be two components, see galaxy.py.

In [161]:
cut_pre_combined = np.ones_like(cut_pre["FLAGS"], dtype=bool)

In [162]:
cut_pre_combined = np.logical_and.reduce(list(cut_pre.values()))

In [163]:
# Output some mask statistics

n_obj = dat.shape[0]

print(f"{'flag':30s} {'n_ok':>10} {'n_ok[%]':>10}")
for name in cut_pre:
    n_ok = sum(cut_pre[name])
    print(f"{name:30s} {n_ok:10d} {n_ok/n_obj:10.2%}")
name = "combined"
n_ok = sum(cut_pre_combined)
print(f"{name:30s} {n_ok:10d} {n_ok/n_obj:10.2%}")

flag                                 n_ok    n_ok[%]
FLAGS                             4835857     82.01%
overlap                           5528786     93.76%
IMAFLAGS_ISO                      5290428     89.72%
N_EPOCH                           5181867     87.88%
mag                               5869604     99.54%
NGMIX_MCAL_FLAGS                  5896744    100.00%
NGMIX_MOM_FAIL                    5875457     99.64%
NGMIX_ELL_PSFo_NOSHEAR_0           108934      1.85%
combined                            54910      0.93%


In [164]:
# Number of "galaxies" (cut_common in main_set_up)

cut_common = cut_pre["overlap"] & cut_pre["FLAGS"] & cut_pre["mag"] & cut_pre["IMAFLAGS_ISO"] & cut_pre["N_EPOCH"]
name = "common"
n_ok = sum(cut_common)
print(f"{name:30s} {n_ok:10d} {n_ok/n_obj:10.2%}")

cut_galaxy = cut_common & cut_pre["NGMIX_MCAL_FLAGS"] & cut_pre["NGMIX_ELL_PSFo_NOSHEAR_0"] & cut_pre["NGMIX_MOM_FAIL"]
name = "galaxy"
n_ok = sum(cut_galaxy)
print(f"{name:30s} {n_ok:10d} {n_ok/n_obj:10.2%}")

common                            3761336     63.79%
galaxy                              54910      0.93%


In [165]:
# Apply post-proc structural masks

cut_combined = cut_pre_combined

### Calibration

In [166]:
# Define cuts and metacal input parameters

# Ellipticity dispersion
sigma_eps_prior = 0.34

# Signal-to-noise range
gal_snr_min = 10
gal_snr_max = 500

# Relative-size (hlr / hlr_psf) range
gal_rel_size_min = 0.5
gal_rel_size_max = 3

# Correct relative size for ellipticity?
gal_size_corr_ell = False

In [170]:
# Call metacal

gal_metacal = metacal(
    dat,
    cut_combined,
    snr_min=gal_snr_min,
    snr_max=gal_snr_max, 
    rel_size_min=gal_rel_size_min,
    rel_size_max=gal_rel_size_max,
    size_corr_ell=gal_size_corr_ell,
    sigma_eps=sigma_eps_prior,
    col_2d=False,
    verbose=True,
)

Metacal cuts: 10<snr<500, rel_size_min=0.5, rel_size_max=3, size_corr_ell=False
Extracting 1M
Extracting 1P
Extracting 2M
Extracting 2P
Extracting NOSHEAR
MKDEBUG  False
Number of objects on metal input = 5896744
Number of objects after galaxy selection masking = 54910


In [179]:
g_corr, g_uncorr, w, mask = calibration.get_calibrated_quantities(gal_metacal)

In [182]:
len(mask)

16318

In [ ]:



# Correct for PSF leakage

# Compute DES weights